In [22]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LLM
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="llama3.2:latest"
)

![Mindmap: Runnable Passthrough and Pydantic](mindmap_runnablepassthrough_pydantic.png)

#### RunnablePassthrough()

This means:

“Take the input and pass it forward unchanged.”<br>
Nothing fancy. No AI. Just forwarding data.<br>
here keep the input text as it is

and then This is the most important concept.

It means:

“From the same input, do TWO things at the same time.”

🔹 Branch 1: original_input<br>
RunnablePassthrough()

- Keeps input exactly as it is

- No modification

🔹 Branch 2: llm_response<br>
prompt | llm | StrOutputParser()


- Sends input to LLM

- Gets clean text output

In [25]:
# Prompt
prompt = ChatPromptTemplate.from_template(
    "Explain this sentence in one line: {text}"
)

In [ ]:

# RunnablePassthrough FIRST, then parallel mapping
chain = (
    RunnablePassthrough()
    | {
        "original_input": RunnablePassthrough(),
        "llm_response": prompt | llm | StrOutputParser()
    }
)

# Invoke
output = chain.invoke({"text": "The sky is blue"})

print(output)


{'original_input': {'text': 'The sky is blue'}, 'llm_response': 'The sentence states that the color of the sky appears as blue.'}


🧠 The BIG CLARITY <br>

❓ “Is RunnablePassthrough used to pass output of one LLM to another?”

✅ Correct understanding now:

- ❌ RunnablePassthrough does NOT call LLM

- ❌ It does NOT transform data

- ✅ It only preserves data

👉 Think of it as:<br>

“Don’t touch this, just carry it forward.”

🧩 Mental Model (Very Simple)

Input
  ↓
Passthrough
  ↓
 ├─ Keep original
 └─ Send to LLM


🔑 Why This Matters (Real Projects)

This pattern is used when you want:<br>

- Audit logs

- Original + AI output

- Routing decisions

- Storing results to DB / CSV

<b>“RunnablePassthrough lets you preserve original input while an LLM processes the same input in parallel. It doesn’t call the LLM — it simply forwards data unchanged.”<b>

### Pydantic Output Parser

What is a Pydantic Output Parser? 

- 👉 Pydantic Output Parser forces the LLM to reply in a fixed structure.

🔹 Why do we even need it?

LLMs normally reply like this:

- “Sure! Here’s the answer…”

But programs need:

{
  "sentiment": "positive",
  "confidence": 0.92
}


👉 Pydantic acts like a strict form the LLM must fill.

🔹 Real-world analogy

Think of a bank form 📝

- Free text ❌ messy

- Fixed fields ✅ reliable

<b>Pydantic = bank form for LLM outputs.


<b>🔹 What Pydantic gives you

- ✅ Predictable fields
- ✅ Type safety (string, int, float)
- ✅ Automatic validation
- ✅ Errors if LLM misbehaves

<b>🔁 Flow (conceptual)

Prompt → LLM → Pydantic Parser → Structured Python Object

- Not text.
- Not guessing.
- Real Python data.


<b>🚫 What Pydantic is NOT

- ❌ It does not generate text
- ❌ It does not improve reasoning
- ❌ It does not fix bad prompts

It only enforces structure.

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        ChatPromptTemplate,
                                        PromptTemplate
                                        )

from typing import  Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


base_url = "http://localhost:11434"
# model = 'qwen3'
model = 'llama3.2:latest'

llm = ChatOllama(base_url=base_url, model=model)

<b>1️⃣ Pydantic Model = Output Contract<b>


This defines what exact shape we want from the LLM.

Think of it as a schema / blueprint.<br>

The LLM must respond with:

- setup

- punchline

- rating

This removes guesswork and messy string parsing.

👉 Why important for beginners?
- You get predictable, typed output, just like APIs.

In [6]:
class Poem(BaseModel):
    """tell Poem to the user"""

    setup: str = Field(description="The setup of the poem")
    punchline: str = Field(description="The punchline of the poem")
    rating: Optional[int] = Field(description="The rating of the poem is from 1 to 5", default=None)

<b>2️⃣ PydanticOutputParser = Enforcer<b>

This parser:

- Tells the LLM how to format the response

- Validates the output

- Converts it into a real Python object

If the LLM output is wrong → parser throws an error (which is GOOD).

In [8]:
parser = PydanticOutputParser(pydantic_object=Poem)

<b>3️⃣ Format Instructions = Teaching the LLM<b>

These instructions are injected into the prompt.

- You are explicitly telling the LLM:

- “Respond ONLY in this JSON format”

This dramatically improves accuracy.

In [9]:
instruction = parser.get_format_instructions()

In [10]:
print(instruction)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "tell Poem to the user", "properties": {"setup": {"description": "The setup of the poem", "title": "Setup", "type": "string"}, "punchline": {"description": "The punchline of the poem", "title": "Punchline", "type": "string"}, "rating": {"anyOf": [{"type": "integer"}, {"type": "null"}], "default": null, "description": "The rating of the poem is from 1 to 5", "title": "Rating"}}, "required": ["setup", "punchline"]}
```


In [11]:
prompt = PromptTemplate(
    template='''
    Answer the user query with a poem. Here is your formatting instruction.
    {format_instruction}

    Query: {query}
    Answer:''',
    input_variables=['query'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)



<b>4️⃣ Prompt + LLM (Normal Generation)<b>

In [12]:
chain = prompt | llm

In [13]:
output = chain.invoke({'query': 'Tell me a poem about the sky'})

At this stage:

- Output is still raw text

- You manually print output.content

In [14]:
print(output.content)

{"setup": "On a clear blue day, so high and wide", "punchline": "The sky meets the eye", "rating": 5}


<b>5️⃣ Prompt + LLM + Parser (Structured Output)<b>

🔥 This is the key moment

The LLM response is:

- Generated

- Validated

- Converted into a Pydantic object

In [17]:
chain = prompt | llm | parser
output = chain.invoke({'query': 'Tell me a poem about the rain'})
print(output)

setup='Rain, rain, falling down\nFrom the sky so high and round' punchline='Bringing life to the earth below, making everything grow.' rating=4
